# 🔎 Chat with a GitHub Repository: Code-Aware RAG

## Learning Objectives
In this notebook, you will learn:
1. **Repository ingestion** - clone a public GitHub repository to disk with GitPython
2. **Language-aware loading and splitting** - load `.py` files with `LanguageParser` and split them on Python syntax boundaries
3. **Code embeddings in Chroma** - embed the code chunks with OpenAI embeddings and persist them in a local Chroma store
4. **Conversational retrieval** - answer questions about the code with a `ConversationalRetrievalChain` that keeps a summary memory

## Where this fits
This is the exploratory build behind the project's Flask app. `src/helper.py`, `store_index.py` and `app.py` package the same steps (clone, split, embed, chat) behind a web UI.

## Prerequisites
- Retrieval-augmented generation basics (repository 05, retrieval and RAG) - as reading only
- Dependencies installed with `uv sync` in this project's folder
- `OPENAI_API_KEY` in this project's `.env` (used for the embeddings and the chat model)
- Internet access: Part 1 clones a public GitHub repository

![image](image.png)

*The pipeline this notebook builds: clone → load and split Python code → embed into Chroma → conversational retrieval.*

---
## 📦 Step 1: Environment Setup

The setup cell imports the libraries, loads the keys from this project's `.env` and suppresses warnings.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports, API keys and warnings
# ============================================================================
import os
import warnings

from dotenv import find_dotenv, load_dotenv
from git import Repo
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationSummaryMemory
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from this project's .env
load_dotenv(find_dotenv(usecwd=True))

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in this project's .env before running this notebook.")

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: Chat model for answers and the summary memory
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📥 Part 1: Clone the Repository

The analyzer works on any public Python repository. Here we clone a small one, a medical chatbot project, into `test_repo/` next to this notebook. The clone runs only when `test_repo/` does not hold a clone yet, so re-running the cell is safe.

> **Key Insight**: Everything downstream reads files from disk, so ingestion is just a `git clone`. The app does the same in `src/helper.py` (`repo_ingestion`).

In [ ]:
# ============================================================================
# CLONE: Fetch a public GitHub repository into test_repo/
# ============================================================================
repo_path = "test_repo/"

# Clone once; a second run reuses the existing clone instead of failing on a non-empty folder
if not os.path.isdir(os.path.join(repo_path, ".git")):
    repo = Repo.clone_from("https://github.com/entbappy/End-to-end-Medical-Chatbot-Generative-AI", to_path=repo_path)

print(f"✅ Repository available at {repo_path}")

---
## ✂️ Part 2: Load and Split the Code by Language

`GenericLoader` walks the folder and keeps only `.py` files. `LanguageParser` parses each file as Python, so top-level functions and classes become separate documents. The splitter then cuts long documents on Python boundaries (`class`, `def`, ...) instead of at arbitrary characters.

### Key Concepts:
- **`parser_threshold=500`**: files shorter than 500 lines are kept whole; longer files are split into functions and classes
- **`RecursiveCharacterTextSplitter.from_language`**: chunking with language-specific separators

In [ ]:
# ============================================================================
# LOAD: Read every .py file in the clone as a parsed Python document
# ============================================================================
loader = GenericLoader.from_filesystem(repo_path,
                                        glob = "**/*",
                                       suffixes=[".py"],
                                       parser = LanguageParser(language=Language.PYTHON, parser_threshold=500)
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
documents

In [ ]:
# ============================================================================
# SPLIT: Cut the documents into ~500-character chunks on Python boundaries
# ============================================================================
documents_splitter = RecursiveCharacterTextSplitter.from_language(language = Language.PYTHON,
                                                             chunk_size = 500,
                                                             chunk_overlap = 20)

texts = documents_splitter.split_documents(documents)

print(f"Split into {len(texts)} chunks")
texts

Each chunk keeps its `source` file path (and, for parsed code, `content_type`) in `metadata`. That is what lets an answer point back to a file.

---
## 🧮 Part 3: Embed the Chunks into Chroma

Each chunk is embedded with OpenAI embeddings and stored in a Chroma database persisted to `./db`. `disallowed_special=()` lets the tokenizer accept code that contains special-token text such as `<|endoftext|>`.

> **Note**: `Chroma.from_documents` adds to an existing `./db`. Re-running this cell stores the chunks again; delete `research/db/` first for a clean index.

In [ ]:
# ============================================================================
# EMBED + STORE: OpenAI embeddings into a persisted Chroma store
# ============================================================================
embeddings = OpenAIEmbeddings(disallowed_special=())

vectordb = Chroma.from_documents(texts, embedding=embeddings, persist_directory='./db')
vectordb.persist()  # no-op on chromadb >= 0.4 (it persists automatically); kept for older versions

print(f"✅ Stored {vectordb._collection.count()} vectors in ./db")

---
## 💬 Part 4: Ask Questions with Conversational Retrieval

`ConversationalRetrievalChain` turns each question (plus the chat history) into a standalone query, retrieves code chunks with MMR (`k=3`, diverse rather than near-duplicate chunks) and answers from them. `ConversationSummaryMemory` keeps a running LLM-written summary of the conversation as the chat history.

### 4.1 🔗 Build the chain

In [ ]:
# ============================================================================
# CHAIN: Summary memory + MMR retriever + chat model
# ============================================================================
memory = ConversationSummaryMemory(llm=llm, memory_key = "chat_history", return_messages=True)
qa = ConversationalRetrievalChain.from_llm(llm, retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k":3}), memory=memory)

print("✅ Conversational retrieval chain ready")

### 4.2 ❓ Ask about functions in the cloned repository

Both questions name functions defined in the cloned repository's code. The answers should describe what that code does, based on the retrieved chunks, not general knowledge about embeddings or text splitting.

In [ ]:
# ============================================================================
# QUESTION 1: What does download_hugging_face_embeddings do?
# ============================================================================
question = "what is download_hugging_face_embeddings function doing?"

result = qa.invoke({"question": question})
print(result['answer'])

In [ ]:
# ============================================================================
# QUESTION 2: What does text_split do?
# ============================================================================
question = "what is text_split function doing?"

result = qa.invoke({"question": question})
print(result['answer'])

The full result also carries the `chat_history` that the summary memory maintains. After two questions it holds a summary of both exchanges.

In [ ]:
# ============================================================================
# INSPECT: The full result, including the summarized chat history
# ============================================================================
result

In [ ]:
# ============================================================================
# QUESTION 3: Re-ask about the first function, phrased differently
# ============================================================================
question = "what is the use of download_hugging_face_embeddings?"

result = qa.invoke({"question": question})
print(result['answer'])

---
## 📝 Summary

In this notebook, we learned:

### 1. Repository ingestion
- A `git clone` (GitPython `Repo.clone_from`) is all the ingestion a code assistant needs.

### 2. Language-aware loading and splitting
- `LanguageParser` and `RecursiveCharacterTextSplitter.from_language` keep functions and classes intact, so a retrieved chunk is a meaningful unit of code.

### 3. Code embeddings in Chroma
- OpenAI embeddings with `disallowed_special=()` handle source code safely; Chroma persists the index to `./db`.

### 4. Conversational retrieval
- `ConversationalRetrievalChain` with MMR retrieval answers questions from the code, and `ConversationSummaryMemory` carries a summarized history across turns.

### Next Steps
- **The project's app**: `store_index.py` and `app.py` (run with `uv run python app.py`, see the project README) package these steps behind a Flask UI where you paste any repository URL and chat with it.